# Session 2 — Build an Agentic RAG System

In this session you will build, step by step, an agent that answers
questions about NVIDIA's financial results using the company's own filings.

The pipeline has four stages:

1. **Chunk** 
2. **Vectorize** 
3. **Search** 
4. **Assemble an agent** 

The corpus is in `data/02_processed/`: NVIDIA's 10-K filings for fiscal 2025 and 2026,
trimmed to their substantive pages, plus the transcripts of the two matching earnings
calls.

Before starting, take a look at the documents themselves in `data/01_raw/` and
`data/02_processed/`. That is the content everything here relies on.

## Setup

The helpers used in this notebook are given to you in `utils/build.py`. Open it and have a
look before going further.

In [ ]:
import json
from pathlib import Path

from utils.build import build_chunk_records, load_document, save_chunks

PROCESSED_DIR = Path("../data/02_processed")
CHUNKS_DIR = Path("../data/03_chunks")
ALL_CHUNKS_PATH = CHUNKS_DIR / "all_chunks.json"

sorted(p.name for p in PROCESSED_DIR.iterdir() if p.suffix in {".pdf", ".txt"})

# Part 1 — Build the retrieval pipeline

The first three exercises turn the corpus into something searchable: cut the documents into
chunks, turn each chunk into a vector, and find the chunks closest to a question.

## Exercise 1 — Chunking the documents

The first step of a RAG system is cutting the corpus into small pieces. How much a model can
read at once depends on the model, and the longer the context, the more it loses track of what
matters. Two parameters set the trade-off:

- **`chunk_size`**: maximum characters in a chunk. Too large wastes tokens and buries the
  answer among irrelevant text; too small truncates the information.
- **`overlap`**: characters shared by consecutive chunks. Without it, a sentence cut by a
  boundary is whole in neither.

With `chunk_size=30` and `overlap=8`, each chunk starts 22 characters after the previous
one and repeats its last 8:

![Chunks of 30 characters overlapping by 8](assets/chunking-overlap.png)

**Objective:** Implement the chunking logic: splitting a document's text into overlapping
pieces.

**Your task:** Complete `chunk_string` below, respecting its signature and the intent
documented in its docstring.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter


def chunk_string(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into overlapping, non-blank chunks.

    Parameters
    ----------
    text : str
        The text to split.
    chunk_size : int
        Maximum number of characters in a chunk.
    overlap : int
        Number of characters each chunk shares with the previous one.

    Returns
    -------
    list[str]
        The chunks, in order. Blank chunks are dropped.
    """
    # YOUR CODE HERE

**Hint:** Use [`CharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/CharacterTextSplitter)
from `langchain_text_splitters`. 

### Check your implementation

In [ ]:
chunks = chunk_string("abcde, fghij. klmn\n opqrs!", chunk_size=10, overlap=3)

assert chunks == ["abcde, fgh", "fghij. klm", "klmn\n opqr", "pqrs!"]
print("OK:", len(chunks), "chunks")

### Chunk every document

The next step is to orchestrate chunking across a list of documents. It proceeds as follows:

1. `load_document(path)` returns the full text of a document along with the fiscal year it
   covers.
2. `chunk_string`, the function you have just written, cuts that text into overlapping pieces.
3. `build_chunk_records(text_chunks, document_name, year)` gives each chunk an `id` and records
   which document and year it came from.
4. `save_chunks(records, path)` writes those records to a JSON file, one per document, in
   `data/03_chunks/`.

In [ ]:
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

for document_path in sorted(PROCESSED_DIR.iterdir()):
    if document_path.suffix not in {".pdf", ".txt"}:
        continue  # skip anything that is not one of our documents

    text, year = load_document(str(document_path))
    text_chunks = chunk_string(text, chunk_size=1000, overlap=200)
    records = build_chunk_records(text_chunks, document_path.name, year)

    save_chunks(records, str(CHUNKS_DIR / f"{document_path.stem}.json"))
    print(f"{document_path.name}: {len(records)} chunks")

### Gather them into one index

Retrieval runs against a single index covering the whole corpus. The cell below reads the
per-document files back and concatenates them into `all_chunks.json`, which every later step
works from.

In [ ]:
all_chunks = []

for document_chunks_path in sorted(CHUNKS_DIR.glob("*.json")):
    if document_chunks_path != ALL_CHUNKS_PATH:  # do not read the gathered file into itself
        all_chunks.extend(json.loads(document_chunks_path.read_text()))

save_chunks(all_chunks, str(ALL_CHUNKS_PATH))
print(f"{len(all_chunks)} chunks in {ALL_CHUNKS_PATH}")

## Exercise 2 — Vectorizing the chunks

A question rarely uses the same words as the passage that answers it. For example, a filing
writes "revenue" where the question says "sales". 

To tackle this problem, an embedding model maps a text to a vector of
numbers, positioned so that texts carrying a similar meaning land close together. Comparing
two vectors then tells you how close two passages are in meaning.

The model below is a small one, which runs locally on your CPU. The first run downloads it
(90 MB).

The vectors it produces are normalised, which is what makes two of them comparable.

In [ ]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from numpy.typing import NDArray

from utils.build import save_vectors

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
    show_progress=True,
)

print("vector dimension:", len(embedding_model.embed_query("test")))

**Objective:** Turn a list of texts into the matrix holding one vector per text.

**Your task:** Complete `embed_texts` below, respecting its signature and the intent
documented in its docstring.

In [ ]:
def embed_texts(texts: list[str], model: HuggingFaceEmbeddings) -> NDArray[np.float32]:
    """Turn each text into a vector.

    Parameters
    ----------
    texts : list[str]
        The texts to embed.
    model : HuggingFaceEmbeddings
        The embedding model, already loaded.

    Returns
    -------
    NDArray[np.float32]
        One row per text, of shape (len(texts), embedding dimension).
    """
    # YOUR CODE HERE

**Hint:** [`embed_documents`](https://reference.langchain.com/python/langchain-huggingface/embeddings/langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings)
embeds a whole list of texts in one call and returns a list of lists. `np.asarray` turns that
into a matrix.

### Check your implementation

In [ ]:
examples = ["revenue grew sharply", "sales increased a lot", "the cafeteria serves lunch"]
vectors = embed_texts(examples, embedding_model)

assert vectors.shape == (3, 384)
np.testing.assert_allclose(
    vectors[:, :3],
    [[0.031227, -0.031101, 0.004742], [-0.034080, -0.028074, 0.002028], [-0.046444, 0.116769, 0.036383]],
    atol=1e-5,
)
print("OK:", vectors.shape[0], "vectors of dimension", vectors.shape[1])

### Vectorize the whole corpus

The cell below embeds every chunk of `all_chunks.json` and writes the result to
`data/04_vectors/`. `save_vectors` stores the matrix together with the chunk ids it came from,
so that row `i` can always be traced back to the chunk it encodes.

In [ ]:
%%time
VECTORS_PATH = Path("../data/04_vectors/chunk_vectors.npz")
VECTORS_PATH.parent.mkdir(parents=True, exist_ok=True)

all_chunks = json.loads(ALL_CHUNKS_PATH.read_text())
vectors = embed_texts([chunk["text"] for chunk in all_chunks], embedding_model)

save_vectors(vectors, [chunk["id"] for chunk in all_chunks], str(VECTORS_PATH))
print(f"{vectors.shape[0]} vectors of dimension {vectors.shape[1]} in {VECTORS_PATH}")

## Exercise 3 — Searching the chunks

The corpus is now a matrix of vectors, and a question can be turned into a vector the same
way. Searching means comparing the question's vector against every chunk's vector and keeping
the closest ones, which become the passages the agent reads before it answers.

Because the vectors are normalised, the dot product of two of them is exactly their cosine
similarity. A score near 1 means the two texts are close in meaning, and a score near 0 means
they have little to do with each other.

**Objective:** Find the `k` chunks whose vectors are closest to the vector of a question.

**Your task:** Complete `top_k_search` below, respecting its signature and the intent
documented in its docstring.

In [ ]:
def top_k_search(query_vector: np.ndarray, chunk_vectors: np.ndarray, k: int) -> list[int]:
    """Find the k chunks whose vectors are closest to the query vector.

    Parameters
    ----------
    query_vector : np.ndarray
        The vector of the question, of shape (embedding dimension,).
    chunk_vectors : np.ndarray
        One row per chunk, of shape (number of chunks, embedding dimension).
    k : int
        Number of chunks to return.

    Returns
    -------
    list[int]
        The row indices of the k closest chunks, closest first.
    """
    # YOUR CODE HERE

**Hint:** The whole search is two numpy operations over the matrix, with no Python loop.
The second needs the function that returns sorting indices instead of sorted values, and
the ids of the chunks matter here rather than the scores themselves.

### Check your implementation

In [ ]:
chunk_vectors = np.array([[0.8, 0.6], [0.0, 1.0], [1.0, 0.0]])
query_vector = np.array([1.0, 0.0])  # scores against the three rows: 0.8, 0.0 and 1.0

assert top_k_search(query_vector, chunk_vectors, k=2) == [2, 0]
print("OK: 2 chunks")

### Search the whole corpus

Retrieval is what gives a model something to work from. The cell below takes a question,
finds the chunks closest to it, and those passages are the context an LLM would read before
answering.

`load_vectors` reads back the matrix written earlier, along with the chunk ids in the same
order.

`top_k_search` returns row numbers, not passages. The rows were saved in the order of
`all_chunks`, so row 7 holds the vector of `all_chunks[7]`, and each row number can be used
directly to look its chunk back up.

In [ ]:
from utils.build import load_vectors

chunk_vectors, chunk_ids = load_vectors(str(VECTORS_PATH))
question = "How does NVIDIA describe its long-term opportunity in physical AI and robotics?"
query_vector = embed_texts([question], embedding_model)[0]

for rank, index in enumerate(top_k_search(query_vector, chunk_vectors, k=3), start=1):
    print(f"[{rank}] {chunk_ids[index]}")
    print(all_chunks[index]["text"][:200], "\n")

**Note:** The matrix and the chunks it encodes, held together and queried by similarity, are
what is called a vector store. Ours keeps everything in memory and compares the question
against every chunk in turn, which is fine for a corpus this size.

A production system uses a dedicated one, such as FAISS, pgvector, Pinecone or Chroma. It
indexes the vectors so a query reaches its nearest neighbours without scanning them all, an
approximate search that keeps latency low over millions of chunks.

# Part 2 — Build the agent

Everything so far runs in a fixed order that you control: chunk, vectorize, search. You decide
when to search, and the search always happens exactly once.

An agent works differently. You hand the model a set of tools and it decides for itself whether
to use them, how to phrase the query, whether to search a second time after reading the first
result, and whether it has enough to answer at all.

### Four words to know first

A **tool** is a function the model is allowed to call. It never sees the body, only the name,
the signature and the docstring.

The **state** is what travels through the agent from step to step. Ours holds one thing: the
list of messages exchanged so far.

A **node** is a function that takes the state and returns what to add to it. Ours are two: one
calls the model, one runs the tools.

An **edge** says what runs next. A plain edge always leads to the same place, while a
**conditional edge** looks at the state and decides.

### What happens when you ask a question

Asking "What was NVIDIA's revenue in fiscal 2026?" sends the agent round this path:

| Step | What runs | Result |
| --- | --- | --- |
| 1 | the state starts | one message, your question |
| 2 | node **model** | the model replies: call `search_filings("NVIDIA fiscal 2026 revenue")` |
| 3 | **conditional edge** | a tool was requested, so go to the tools node |
| 4 | node **tools** | the search runs and its passages are added as a new message |
| 5 | edge | always back to the model |
| 6 | node **model** | reading three messages now, it replies with text and asks for nothing |
| 7 | **conditional edge** | no tool requested, so stop |

Steps 3 to 6 are the loop. On a harder question the model can go round twice, searching again
with different wording.

This is also why the state uses a reducer called `add_messages`. Each node returns only the
message it just produced, and the reducer appends it to the conversation. Without it, step 4
would replace the conversation instead of extending it.

### Setup for this part

The imports for the agent, gathered in one place. `langgraph` provides the graph, and
`langchain_anthropic` the model.

In [ ]:
from collections.abc import Callable

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from utils.build import State, make_answer_or_call_tool

## Exercise 4 — Giving the model its tools

`top_k_search` speaks vectors and row numbers, and a model speaks text, so the retrieval has to
be wrapped in something the model can call.

The `@tool` decorator does that. It exposes a function to the model, which sees its name, its
signature and its docstring, and nothing else. The docstring is therefore what the model reads
to decide when to call the tool and what to pass it.

**Objective:** turn the retrieval pipeline into something the model can call.

**Your task:** complete `search_filings` below, respecting its signature and the intent
documented in its docstring.

In [ ]:
@tool
def search_filings(query: str, k: int = 5) -> str:
    """Search NVIDIA's filings and earnings calls for passages relevant to a question.

    Parameters
    ----------
    query : str
        What to look for, in plain English.
    k : int, optional
        Number of passages to return, by default 5.

    Returns
    -------
    str
        The matching passages, each preceded by the id of the chunk it comes from.
    """
    # YOUR CODE HERE

**Hint:** the three steps are the ones you have already written. Turn the query into a vector
with `embed_texts`, find the closest rows with `top_k_search`, and join the matching chunks
into one string. `embedding_model`, `chunk_vectors` and `all_chunks` are already loaded above.

### Check your implementation

In [ ]:
# A decorated tool is called with .invoke() and a dictionary of arguments.
passages = search_filings.invoke({"query": "physical AI and robotics", "k": 3})

assert passages.count("[") >= 3
assert "robot" in passages.lower()
print("OK:", len(passages), "characters of context")

### A second tool

We are working on financial documents, so the agent also needs a simple calculator.

**Objective:** give the model a way to compute rather than guess.

**Your task:** complete `calculator` below, respecting its signature and the intent documented
in its docstring.

In [ ]:
@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Apply an arithmetic operation to two numbers.

    Parameters
    ----------
    a : float
        The left operand.
    b : float
        The right operand.
    operation : str
        One of "add", "subtract", "multiply", "divide".

    Returns
    -------
    float
        The result of the operation.
    """
    # YOUR CODE HERE

### Check your implementation

In [ ]:
assert calculator.invoke({"a": 215938, "b": 130497, "operation": "subtract"}) == 85441
assert calculator.invoke({"a": 3, "b": 4, "operation": "multiply"}) == 12
print("OK: revenue grew by $85,441 million")

### The model and its tools

Now let's create the model and bind the two tools to it, so that it knows they exist.

`State` is what travels through the agent, and it is imported from `utils.build` along with the
node that calls the model.

In [ ]:
load_dotenv()

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)

# Bind our tools to the LLM
llm_with_tools = llm.bind_tools([search_filings, calculator])
answer_or_call_tool = make_answer_or_call_tool(llm_with_tools)

print("model ready:", llm.model)

## Exercise 5 — Assembling the loop

The pieces are ready and what remains is to connect them. Two nodes are registered under a
name, then the edges say how control moves: into the graph, out of the model node depending on
what it replied, and back from the tools to the model.

**Objective:** assemble the nodes and edges into a runnable agent.

**Your task:** complete `make_agentic_rag` below. Each line is started for you, and the comment
beside it says what it should do.

In [ ]:
def make_agentic_rag(node: Callable[[State], State], tools: list[BaseTool]) -> CompiledStateGraph:
    """Wire the model and its tools into a ReAct loop.

    Parameters
    ----------
    node : Callable[[State], State]
        The node that calls the model.
    tools : list[BaseTool]
        The tools the model is allowed to call.

    Returns
    -------
    CompiledStateGraph
        The compiled agent, ready to be invoked.
    """
    graph = StateGraph(State)

    graph.add_node(..., ...)  # register the model node, under the name "answer_or_call_tool"
    graph.add_node(..., ...)  # register the node that runs the tools, under the name "tools"
    graph.add_edge(..., ...)  # enter the graph on the model node
    graph.add_conditional_edges(..., ..., ...)  # go to the tools if one was asked for, else stop
    graph.add_edge(..., ...)  # close the loop: once the tools have run, back to the model

    return graph.compile()

**Hints:** `add_node` takes a name and a function, and `add_edge` a source and a target, with
`START` and `END` for the entry and the exit.

`add_conditional_edges` takes a source, a router and a mapping. `tools_condition` is the
router: it inspects the last message and answers either `"tools"` or `END`, which the mapping
turns into the node to run next.

### Check your implementation

In [ ]:
agent = make_agentic_rag(answer_or_call_tool, [search_filings, calculator])

result = agent.invoke({"messages": [HumanMessage("What does NVIDIA say about physical AI?")]})
message_types = [type(message).__name__ for message in result["messages"]]

assert "ToolMessage" in message_types
print("OK:", " -> ".join(message_types))

### Watch the loop turn

The check above showed the message types, and it is worth reading the messages themselves. The
cell below prints the whole conversation: your question, what the model asked for, what the
tool returned, and the answer it settled on.

The system prompt is what tells the agent to search before answering, to cite the chunks it
used, and to say so when the passages do not contain the answer.

In [ ]:
SYSTEM_PROMPT = (
    "You answer questions about NVIDIA using the company's own filings and earnings calls. "
    "Search the filings before answering a question about the company. "
    "Base your answer only on the passages you retrieve, and cite the chunk ids you used. "
    "If the passages do not contain the answer, say so plainly instead of guessing. "
    "If a question makes no sense, say so."
)


def show_conversation(question: str, agent: CompiledStateGraph) -> None:
    """Run the agent on one question and print every message it produced."""
    result = agent.invoke({"messages": [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]})

    for message in result["messages"]:
        label = {HumanMessage: "HUMAN", AIMessage: "AI", ToolMessage: "TOOL"}.get(type(message))
        if label is None:
            continue
        print(f"\n[{label}] {str(message.content)[:400]}")
        for tool_call in getattr(message, "tool_calls", []):
            print(f"  calls {tool_call['name']} with {tool_call['args']}")


show_conversation("How did NVIDIA's revenue change between fiscal 2025 and fiscal 2026?", agent)

### Three things worth checking

A useful agent has to do more than answer the easy questions. The three below come from
`questions.json` and were written for this, one per category:

1. A question the filings answer, where you can check both the figure and the chunks cited.
2. A question that sounds answerable but is not covered by these documents, where the agent
   should say so rather than invent a figure.
3. A question built on a false premise, which the agent should notice instead of playing along.

`answer_question` below is given: it runs the agent and returns only its final message, which
is what you want once the loop itself is no longer the thing being inspected.

In [ ]:
def answer_question(question: str, agent: CompiledStateGraph) -> str:
    """Ask the agent one question and return its final answer.

    Parameters
    ----------
    question : str
        The question to ask.
    agent : CompiledStateGraph
        The compiled agent, as returned by `make_agentic_rag`.

    Returns
    -------
    str
        The text of the agent's last message.
    """
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]
    return agent.invoke({"messages": messages})["messages"][-1].content


questions = json.loads(Path("../data/05_questions/questions.json").read_text())["questions"]
by_id = {question["id"]: question for question in questions}

for question_id in (3, 17, 20):
    question = by_id[question_id]
    print(f"\n{'=' * 78}\n[{question['category']}] {question['question']}\n{'=' * 78}")
    print(answer_question(question["question"], agent))

### What you have built, and what it gets wrong

The agent works, and it is worth being precise about how well.

Retrieval is the weak link. The embedding model compares meanings, which serves a question
about strategy or risk well, and serves a question about a figure in a table badly. Asked for a
revenue number, it readily returns a passage about deferred revenue, because the two read alike.
When the agent answers vaguely or says it cannot find something that is plainly in the filings,
this is usually why, and not a mistake in your code.

Three things would improve it, in rough order of effect. Searching on words as well as on
meaning would catch the figures that the embeddings miss. Cutting the documents along their
structure rather than every thousand characters would stop tables being separated from their
headers. Letting the agent reformulate and search again, rather than accepting its first
attempt, would recover some of the rest.